In [1]:
import itertools
import matplotlib; matplotlib.use("TkAgg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import random
from string import Template

In [2]:
def color_generator(size):
    rand_color = []
    for j in range(size):
        rand_color.append("#"+''.join([random.choice('ABCDEF0123456789') for i in range(6)]))
    return rand_color

In [3]:
def update(counter):
    epoch_number =  frame_data[counter,0]
    iter_number = frame_data[counter,1]
    # update route data
    for v in range(nbVehicle):
        epoch_nodes = currResults[v][currResults[v][:,0] == epoch_number]
        
        iter_IDs = epoch_nodes[epoch_nodes[:,1] == iter_number]
        iter_nodes = np.array([locations[index] for index in (iter_IDs[:,6]).astype(int)])


        completed_IDs = compResults[v][compResults[v][:,0] == epoch_number]
        completed_nodes = np.array([locations[index] for index in (completed_IDs[:,5]).astype(int)])
        
        comp_routes[v].set_data(completed_nodes[:,1], completed_nodes[:,2])
        curr_routes[v].set_data(iter_nodes[:,1], iter_nodes[:,2])
    
    # update scatter data 
    data = request_data[request_data[:,4] <= simulStart + epoch_number * EpochLength]
    available = data[data[:,5] > simulStart + (epoch_number+1) * EpochLength]
    completed = data[data[:,6] <= simulStart + (epoch_number+1) * EpochLength]
    onboard = data[data[:,5] <= simulStart + (epoch_number+1) * EpochLength]
    onboard = onboard[onboard[:,6] > simulStart + (epoch_number+1) * EpochLength]
    
    
    for v in range(nbVehicle):
        vehicle_onboard = onboard[onboard[:,7] == v]
        planned = onboard[onboard[:,7] != v]
        remained = np.concatenate([available, vehicle_onboard])
        remained_IDs = np.concatenate([available, vehicle_onboard])
        remained_pick = np.array([locations[index] for index in (remained_IDs[:,2]).astype(int)])
        remained_drop = np.array([locations[index] for index in (remained_IDs[:,3]).astype(int)])
        
        unavailable = np.concatenate([completed, planned])
        unavailable_IDs = np.concatenate([completed, planned])
        unavailable_pick = np.array([locations[index] for index in (unavailable_IDs[:,2]).astype(int)])
        unavailable_drop = np.array([locations[index] for index in (unavailable_IDs[:,3]).astype(int)])
        if np.shape(unavailable_pick) == (0,):
            unavailable_pick = np.empty([0,2], dtype= 'float64')
            
        if np.shape(unavailable_drop) == (0,):
            unavailable_drop = np.empty([0,2], dtype= 'float64')
            
        if np.shape(remained_pick) == (0,):
            remained_pick = np.empty([0,2], dtype= 'float64')
        
        if np.shape(remained_drop) == (0,):
            remained_drop = np.empty([0,2], dtype= 'float64')
       
        
        scats[v][0].set_offsets(unavailable_pick[:,1:3])
        scats[v][1].set_offsets(unavailable_drop[:,1:3])
        scats[v][2].set_offsets(remained_pick[:,1:3])
        scats[v][3].set_offsets(remained_drop[:,1:3])
        
        
        scats[v][0].set_edgecolors([point_colors[index] for index in (unavailable[:,0]).astype(int)])
        scats[v][1].set_edgecolors([point_colors[index] for index in (unavailable[:,0]).astype(int)])
        scats[v][2].set_facecolors([point_colors[index] for index in (remained[:,0]).astype(int)])
        scats[v][3].set_facecolors([point_colors[index] for index in (remained[:,0]).astype(int)])

        

    
    title.set_text(title_template.substitute(starttime= str((epoch_number)*EpochLength), 
                                             endtime= str((epoch_number+1)*EpochLength), 
                                             epochs= str(epoch_number), 
                                             iters = str(iter_number)))
    return scats

# MAIN   PROGRAM

In [4]:
frame_length = 1
instance_Name = "20150713_16-03m"
results_date = "20220416-0243"
datapath = "DARP_IPS/datasets/" 
nbVehicle = 6
speedInterval = 2000
EpochLength = 50
simulStart = 57600

polygon_filename = "Manhattan_polygon_neigh1-6.csv"
requests_filename = datapath + instance_Name + "/Results_" + results_date + "/Requests_" + instance_Name + ".csv"
compRoutes_filename = datapath + instance_Name+ "/Results_" + results_date + "/finalSolution_" + instance_Name + ".csv"
currRoutes_filename = datapath + instance_Name + "/Results_" + results_date + "/epochSolution_" + instance_Name + ".csv"
locations_filename = datapath + instance_Name + "/LOCATION_" + instance_Name + ".csv"


### Reading data

In [5]:
df_requests = pd.read_csv(requests_filename, sep=',', header = 0)
df_polygon = pd.read_csv(polygon_filename, sep=',', header = 0)
df_currRoutes = pd.read_csv(currRoutes_filename, sep=',', header = 0)
df_compRoutes = pd.read_csv(compRoutes_filename, sep=',', header = 0)
df_locations = pd.read_csv(locations_filename, sep=',', header = 0)

### Preprocess data

In [6]:
polygon_data =df_polygon.to_numpy(dtype='double')
request_data =df_requests.to_numpy(dtype='double')
comp_data = df_compRoutes.drop(columns =['NodeID']).to_numpy(dtype='double')
curr_data = df_currRoutes.drop(columns =['NodeID']).to_numpy(dtype='double')
frame_data = df_currRoutes[['Epoch', ' ISUDIter']].drop_duplicates().to_numpy(dtype='int')
locations = df_locations.to_numpy(dtype='double')

In [7]:
currResults = []
for v in range(nbVehicle):
    currResults.append(np.array(curr_data[curr_data[:,2] == v]))

compResults = []
for v in range(nbVehicle):
    compResults.append(np.array(comp_data[comp_data[:,1] == v]))

### define colors

In [8]:
point_colors = color_generator(np.shape(request_data)[0])
colorList = ['darkred','seagreen','steelblue', 'darkorange', 'deeppink','darkgoldenrod']
route_color = itertools.cycle(colorList)
complete_color = itertools.cycle(colorList)

### define plots

In [9]:
fig = plt.figure()
gs = fig.add_gridspec(2, 3, hspace=0, wspace=0)
axs = gs.subplots(sharex='col', sharey='row')
fig.tight_layout()

title_template = Template('time = ${starttime} - ${endtime} (s)       solution epoch = ${epochs}       ISUD iteration = ${iters}')
title = plt.suptitle(t='', fontsize = 17, fontname="Garamond", fontweight="bold")

scats = []
comp_routes = []
curr_routes = []
v_counter = 0

for ax in axs.ravel():

    ax.set_xlim(round(min(polygon_data[:,0]),2)-0.01, round(max(polygon_data[:,0]),2)+0.01)
    ax.set_ylim(round(min(polygon_data[:,1]),2)-0.01, round(max(polygon_data[:,1]),2)+0.01)
    ax.set_xlabel('Latitude')
    ax.set_ylabel('Longitude')
    line, = ax.plot(np.array(polygon_data[:,0]), np.array(polygon_data[:,1]) , 'grey')
    source = ax.scatter([locations[compResults[v_counter][0][5].astype(int)][1]], 
                        [locations[compResults[v_counter][0][5].astype(int)][2]], c = "grey", s = 110, marker = ",")

    scats.append([ax.scatter([], [], c= 'w',  s = 40, alpha=1, linewidth = 1.5, edgecolor='y', label="unavailable pickup"),
                  ax.scatter([], [], c= 'w', s = 40, marker = ',', linewidth = 1.5, edgecolor='y', alpha=1, label="unavailable dropoff"),
                  ax.scatter([], [], c= 'c', s = 40, alpha=1, label="pickup"),
                  ax.scatter([], [], c= 'c', s = 40, marker = ',', alpha=1, label="dropoff")])
    ax.legend()
    
    comp_routes.append(ax.plot([], [], linewidth=1, c = next(complete_color))[0])
    
    curr_routes.append(ax.plot([], [], linewidth=3, c = next(route_color))[0])
    v_counter = v_counter + 1

### animation function

In [10]:
ani = animation.FuncAnimation(fig, func = update, frames = np.arange(0,np.shape(frame_data)[0],frame_length), interval = speedInterval, repeat = True)
plt.show()